# RigTech · runner_colab_continuacao — DINOv3 + Transformer (per-crop 3-class)

**Filosofia do ciclo**: modelo é **instrumento fixo de auditoria de rótulos** — mesmo backbone (DINOv3-ViTL16 satélite, congelado), mesma head (2 layers Transformer + mean pool), mesmos hiperparâmetros. Só o **dataset** muda entre ciclos. Artefatos versionados (`_c1.pt`, `_c2.pt`, ...) pra medir o quanto as correções melhoraram o rótulo.

**Task**: classificação **3-class** por **crop 224×224**:
- `0` = cultivo
- `1` = folha_larga
- `2` = folha_estreita

Geojsons de mamona ficam **fora do escopo atual** (ignorados). Se quiser incluir depois, é só editar `GEOJSON_CLASS_MAP` no config.

**Amostragem**:
- **Positivos** (`label=1` ou `2`): 1 crop centrado no centroide de cada polígono anotado — classe deduzida do nome do arquivo geojson.
- **Negativos** (`label=0`, cultivo): até `MAX_CULTIVO_CROPS_PER_SITE` crops sampleados da plantação (respeitando `INVERTED_PLANTACAO` pra Flaviano), sem interseção com nenhuma daninha (inclui mamona no filtro pra não vazar).

**Uso pra achar rótulos ruins** (só sobre positivos anotados):
- `misclass`: modelo prediz classe diferente da anotada (ex: polígono é folha_larga mas modelo diz folha_estreita, ou cultivo) → provavelmente o polígono está errado.
- `low_conf`: `prob_da_classe_verdadeira < SUSPECT_LOW_CONF_THRESHOLD` — geometria ambígua.

**Pré-requisitos no Drive**:
- `MyDrive/Datasets/DaninhasTreinoClientes/{Giasa,DoisRiosFlaviano,Flaviano01,CelsoSTE2,Celso01}/{imagem,daninhas,plantacao}`.
- Secret `HF_TOKEN` no Colab (chave lateral) — `dinov3-vitl16-pretrain-sat493m` é *gated*.

Runtime → GPU **A100** (T4 funciona).

## 1. Montar Drive e instalar dependências

In [1]:
from google.colab import drive
drive.mount('/content/drive')

# Colab as vezes vem com Python 3.13 + pandas/numpy em versoes que quebram
# geopandas ('using_string_dtype' missing, 'partially initialized module pandas'
# etc). Forca versoes compativeis explicitas ANTES do resto.
!pip -q uninstall -y pandas numpy
!pip -q install 'numpy>=1.26,<2.2' 'pandas>=2.2,<2.3'
!pip -q install rasterio geopandas scikit-image joblib tqdm scikit-learn shapely transformers torch huggingface_hub

# IMPORTANTE: apos rodar esta celula, faca Runtime -> Restart runtime (Ctrl+M .)
# antes de rodar as proximas. Pip nao recarrega modulos ja importados na sessao.

Mounted at /content/drive
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.0/62.0 kB 556.5 kB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 89.9/89.9 kB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.0/16.0 MB 51.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.7/12.7 MB 42.7 MB/s eta 0:00:00


## 2. Imports

In [4]:
import os
import csv
import math
import hashlib
import inspect
import numpy as np
import rasterio
from rasterio.windows import Window
import geopandas as gpd
from shapely.geometry import Point
from shapely.ops import unary_union
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from transformers import AutoModel, AutoImageProcessor
import joblib
from tqdm import tqdm
from sklearn.metrics import classification_report, confusion_matrix, f1_score, precision_score, recall_score

## 3. Login no Hugging Face

O checkpoint `dinov3-vitl16-pretrain-sat493m` é gated. Aceite os termos e coloque o token em *Secrets* do Colab como `HF_TOKEN`.

In [5]:
from huggingface_hub import login

_hf_token = None
try:
    from google.colab import userdata
    _hf_token = userdata.get('HF_TOKEN')
except Exception:
    _hf_token = os.environ.get('HF_TOKEN')

if _hf_token:
    login(token=_hf_token)
    print('Login no Hugging Face OK (via HF_TOKEN).')
else:
    login()

Login no Hugging Face OK (via HF_TOKEN).


## 4. Configuração

**Filosofia do ciclo**: NADA aqui muda entre rodadas — só `CICLO` incrementa pra versionar os artefatos.

In [6]:
# ---- ciclo (versiona artefatos, nao altera treino) ----
CICLO = 1

BASE = '/content/drive/MyDrive/Datasets/DaninhasTreinoClientes'

# Sites: (nome, imagem, [geojsons_de_daninha], plantacao). TODOS os geojsons de
# daninha viram classe 1, independente do tipo (folha_larga/folha_estreita/mamona).
PARES_CONFIG = [
    {
        'nome': 'Giasa',
        'imagem': f'{BASE}/Giasa/imagem/Giasa.tif',
        'geojsons': [
            f'{BASE}/Giasa/daninhas/FolhaLargaGiasa (1).geojson',
            f'{BASE}/Giasa/daninhas/FolhaEstreitaGiasa (1).geojson',
            f'{BASE}/Giasa/daninhas/MamonasGiasa (1).geojson',
        ],
        'plantacao': f'{BASE}/Giasa/plantacao/Giasa_plantacao.geojson',
    },
    {
        'nome': 'DoisRiosFlaviano',
        'imagem': f'{BASE}/DoisRiosFlaviano/imagem/DoisRiosFlaviano.tif',
        'geojsons': [
            f'{BASE}/DoisRiosFlaviano/daninhas/FolhaLargaDoisRiosFlaviano.geojson',
            f'{BASE}/DoisRiosFlaviano/daninhas/FolhaEstreitaDoisRiosFlaviano.geojson',
            f'{BASE}/DoisRiosFlaviano/daninhas/MamonasDoisRiosFlaviano.geojson',
        ],
        'plantacao': f'{BASE}/DoisRiosFlaviano/plantacao/DoisRiosFlaviano_plantacao.geojson',
    },
    {
        'nome': 'Flaviano',
        'imagem': f'{BASE}/Flaviano01/imagem/Flaviano01.tif',
        'geojsons': [
            f'{BASE}/Flaviano01/daninhas/FolhaLargaFlaviano-1 (1).geojson',
            f'{BASE}/Flaviano01/daninhas/FolhaEstreitaFlaviano-1 (1).geojson',
            f'{BASE}/Flaviano01/daninhas/MamonasFlaviano-1 (1).geojson',
        ],
        'plantacao': f'{BASE}/Flaviano01/plantacao/Flaviano_plantacao.geojson',
    },
    {
        'nome': 'CelsoSTE2',
        'imagem': f'{BASE}/CelsoSTE2/imagem/CelsoSTE2.tif',
        'geojsons': [
            f'{BASE}/CelsoSTE2/daninhas/FolhaLargaCelsoSTE-2 (1).geojson',
            f'{BASE}/CelsoSTE2/daninhas/FolhaEstreitaCelsoSTE-2 (1).geojson',
            f'{BASE}/CelsoSTE2/daninhas/MamonasCelsoSTE-2 (1).geojson',
        ],
        'plantacao': f'{BASE}/CelsoSTE2/plantacao/CelsoSTE2_plantacao.geojson',
    },
    {
        'nome': 'Celso01',
        'imagem': f'{BASE}/Celso01/imagem/Celso01.tif',
        'geojsons': [
            f'{BASE}/Celso01/daninhas/FolhasLargas_Celso_01 (1).geojson',
            f'{BASE}/Celso01/daninhas/FolhaEstreita_Celso_01 (1).geojson',
        ],
        'plantacao': f'{BASE}/Celso01/plantacao/Celso01_plantacao.geojson',
    },
]

# So o Flaviano tem o poligono de 'plantacao' com semantica invertida (marca a
# area SEM interesse). Confirmado na sessao do notebook base.
INVERTED_PLANTACAO = {'Flaviano'}

# Classe deduzida do nome do arquivo geojson (case-insensitive substring).
# NAO reordenar -- reinterpreta silenciosamente todo o dataset.
GEOJSON_CLASS_MAP = [
    ('folhalarga', 1),
    ('folhaslarga', 1),
    ('folhaestreita', 2),
    ('folhasestreita', 2),
]
# Geojsons contendo qualquer um destes substrings sao IGNORADOS (nao viram
# positivos, mas ainda contam como daninha pra filtrar amostras negativas).
IGNORE_GEOJSON_SUBSTR = ['mamona']
CLASS_NAMES = {0: 'cultivo', 1: 'folha_larga', 2: 'folha_estreita'}
N_CLASSES = 3

# ---- DINOv3 (variante satelite) ----
DINO_MODEL = 'facebook/dinov3-vitl16-pretrain-sat493m'
CROP_PIXELS = 224     # tamanho do crop enviado ao DINOv3 (multiplo de PATCH)
PATCH = 16            # grade 14x14 = 196 tokens por crop

# ---- Amostragem ----
MIN_POLYGON_PIXELS = 25            # descarta slivers minusculos
MAX_CULTIVO_CROPS_PER_SITE = 1500  # crops negativos (cultivo) por site

# ---- Split treino/validacao: faixa espacial dentro de CADA imagem ----
VAL_FRACTION = 0.20
MARGIN_TILES = 1       # gap de 1 x CROP_PIXELS entre treino e val (evita vazamento)
SPLIT_AXIS = 'x'

# ---- Transformer head ----
TRANSFORMER_LAYERS = 2
NHEAD = 8              # 1024 / 8 = 128 por cabeca
TRANSFORMER_DROPOUT = 0.15
CLASSIFIER_DROPOUT = 0.2

EPOCHS = 25
LR = 1e-4
WEIGHT_DECAY = 1e-4
WARMUP_EPOCHS = 2
BATCH = 64
GRAD_CLIP_NORM = 1.0
EARLY_STOP_PATIENCE = 7
RANDOM_STATE = 42

# ---- Loss: peso automatico pra classe minoritaria + label smoothing ----
CLASS_WEIGHTS = None       # None -> calculado por 1/freq no treino
LABEL_SMOOTHING = 0.05

# ---- Suspeitas ----
SUSPECT_LOW_CONF_THRESHOLD = 0.6   # positivo com prob_daninha < isso vira low_conf

# ---- Saidas versionadas por ciclo ----
OUTPUT_MODEL = f'/content/drive/MyDrive/modelo_dinov3_3class_c{CICLO}.pt'
REPORT_PATH  = f'/content/drive/MyDrive/relatorio_dinov3_3class_c{CICLO}.txt'
SUSPECTS_CSV = f'/content/drive/MyDrive/suspeitas_c{CICLO}.csv'
# Cache de features: NAO depende do ciclo. Assinatura por site inclui mtime dos
# geojsons -- se voce mudar um site, so ele recalcula.
CHECKPOINT_DIR = '/content/drive/MyDrive/dinov3_3class_checkpoint'
os.makedirs(CHECKPOINT_DIR, exist_ok=True)
print('Ciclo:', CICLO, '| output:', OUTPUT_MODEL)

Ciclo: 1 | output: /content/drive/MyDrive/modelo_dinov3_3class_c1.pt


## 4b. Snapshot automático do dataset (ciclo N)

Antes do treino, registra o estado atual dos geojsons: SHA256, contagem de polígonos, tamanho, mtime. Salva **manifest JSON** + **tarball dos geojsons** (poucos MBs) em `/content/drive/MyDrive/rigtech_ciclos/`. Se existir manifest do ciclo anterior, imprime **diff**.

Torna cada ciclo **reproduzível** (mesmos geojsons → mesmo modelo) e **auditável** (você vê o que mudou entre versões). Não altera o treino — é só registro.

In [7]:
import json
import tarfile

CICLOS_DIR = '/content/drive/MyDrive/rigtech_ciclos'
os.makedirs(CICLOS_DIR, exist_ok=True)
MANIFEST_PATH = f'{CICLOS_DIR}/dataset_manifest_c{CICLO}.json'
TARBALL_PATH  = f'{CICLOS_DIR}/dataset_geojsons_c{CICLO}.tar.gz'


def _sha256(path):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        for chunk in iter(lambda: f.read(1 << 20), b''):
            h.update(chunk)
    return h.hexdigest()


def _count_polys(path):
    try:
        return int(len(gpd.read_file(path)))
    except Exception:
        return -1


def _file_entry(path):
    if not os.path.exists(path):
        return {'exists': False, 'basename': os.path.basename(path)}
    st = os.stat(path)
    return {
        'exists': True, 'basename': os.path.basename(path),
        'size': st.st_size, 'mtime': int(st.st_mtime),
        'sha256': _sha256(path), 'n_polygons': _count_polys(path),
    }


manifest = {'ciclo': CICLO, 'base': BASE, 'sites': {}}
files_to_snapshot = []
n_polys_total = 0
for cfg in PARES_CONFIG:
    entry = {'geojsons': {}, 'plantacao': None}
    for gj in cfg['geojsons']:
        info = _file_entry(gj)
        entry['geojsons'][os.path.basename(gj)] = info
        if info.get('exists'):
            files_to_snapshot.append(gj)
            n_polys_total += max(info.get('n_polygons', 0), 0)
    if cfg.get('plantacao'):
        entry['plantacao'] = _file_entry(cfg['plantacao'])
        if entry['plantacao'].get('exists'):
            files_to_snapshot.append(cfg['plantacao'])
    manifest['sites'][cfg['nome']] = entry

manifest['n_polygons_daninha_total'] = n_polys_total

with open(MANIFEST_PATH, 'w') as f:
    json.dump(manifest, f, indent=2)
print(f'Manifest: {MANIFEST_PATH} ({n_polys_total} poligonos de daninha, {len(files_to_snapshot)} arquivos)')

if not os.path.exists(TARBALL_PATH):
    with tarfile.open(TARBALL_PATH, 'w:gz') as tar:
        for gj in files_to_snapshot:
            tar.add(gj, arcname=os.path.relpath(gj, BASE))
    print(f'Tarball: {TARBALL_PATH}')
else:
    print(f'Tarball ja existe (nao sobrescrevi): {TARBALL_PATH}')

# ---- diff vs ciclo anterior ----
prev_manifest_path = f'{CICLOS_DIR}/dataset_manifest_c{CICLO - 1}.json'
if os.path.exists(prev_manifest_path):
    with open(prev_manifest_path) as f:
        prev = json.load(f)
    print(f'\n=== Diff vs ciclo {CICLO - 1} ===')
    n_changes = 0
    for site, entry in manifest['sites'].items():
        prev_entry = prev['sites'].get(site, {'geojsons': {}, 'plantacao': None})
        for name, curr in entry['geojsons'].items():
            prev_info = prev_entry.get('geojsons', {}).get(name, {'exists': False})
            if not prev_info.get('exists') and curr.get('exists'):
                print(f'  + [novo]        {site}/{name} ({curr.get("n_polygons", 0)} polys)')
                n_changes += 1
            elif prev_info.get('exists') and not curr.get('exists'):
                print(f'  - [removido]    {site}/{name}')
                n_changes += 1
            elif prev_info.get('sha256') and prev_info.get('sha256') != curr.get('sha256'):
                dp = curr.get('n_polygons', 0) - prev_info.get('n_polygons', 0)
                print(f'  ~ [modificado]  {site}/{name} polygons {prev_info.get("n_polygons")} -> {curr.get("n_polygons")} ({dp:+d})')
                n_changes += 1
        curr_pl = entry.get('plantacao')
        prev_pl = prev_entry.get('plantacao')
        if curr_pl and prev_pl and curr_pl.get('sha256') != prev_pl.get('sha256'):
            print(f'  ~ [modificado]  {site}/{curr_pl["basename"]} (plantacao)')
            n_changes += 1
    prev_total = prev.get('n_polygons_daninha_total', '?')
    print(f'\ntotal poligonos daninha: {prev_total} -> {n_polys_total}')
    if n_changes == 0:
        print("nenhuma mudanca -- rodar de novo produzira o MESMO modelo (bump CICLO so faz sentido se algo mudou).")
else:
    print(f"\n(ciclo {CICLO} e o primeiro -- sem diff pra imprimir)")

Manifest: /content/drive/MyDrive/rigtech_ciclos/dataset_manifest_c1.json (6698 poligonos de daninha, 19 arquivos)
Tarball ja existe (nao sobrescrevi): /content/drive/MyDrive/rigtech_ciclos/dataset_geojsons_c1.tar.gz

(ciclo 1 e o primeiro -- sem diff pra imprimir)


## 5. Carregar DINOv3 (congelado)

Backbone congelado (`eval`, `no_grad`, na GPU). Normalização lida do `AutoImageProcessor` do checkpoint satélite. `interpolate_pos_encoding=True` pra tolerar variação do grid.

In [8]:
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Dispositivo:', DEVICE, (torch.cuda.get_device_name(0) if DEVICE == 'cuda' else ''))

proc = AutoImageProcessor.from_pretrained(DINO_MODEL)
DINOV3_MEAN = [float(x) for x in proc.image_mean]
DINOV3_STD  = [float(x) for x in proc.image_std]
print('mean:', DINOV3_MEAN, ' std:', DINOV3_STD)

print('Carregando DINOv3 (congelado):', DINO_MODEL)
model = AutoModel.from_pretrained(DINO_MODEL)
model.eval()
for p in model.parameters():
    p.requires_grad = False
model = model.to(DEVICE)

PATCH_SIZE = model.config.patch_size
NUM_REGISTER_TOKENS = getattr(model.config, 'num_register_tokens', 0)
HIDDEN_SIZE = model.config.hidden_size
print(f'patch_size={PATCH_SIZE} hidden_size={HIDDEN_SIZE} num_register_tokens={NUM_REGISTER_TOKENS}')
assert PATCH_SIZE == PATCH
assert CROP_PIXELS % PATCH == 0

mean_t = torch.tensor(DINOV3_MEAN, device=DEVICE).view(1, 3, 1, 1)
std_t  = torch.tensor(DINOV3_STD,  device=DEVICE).view(1, 3, 1, 1)
SUPPORTS_INTERP = 'interpolate_pos_encoding' in inspect.signature(model.forward).parameters
N_TOKENS = (CROP_PIXELS // PATCH) ** 2
print(f'crop {CROP_PIXELS}x{CROP_PIXELS} -> grid {CROP_PIXELS//PATCH}x{CROP_PIXELS//PATCH} = {N_TOKENS} tokens')

Dispositivo: cuda NVIDIA L4


preprocessor_config.json:   0%|          | 0.00/585 [00:00<?, ?B/s]

mean: [0.43, 0.411, 0.296]  std: [0.213, 0.156, 0.143]
Carregando DINOv3 (congelado): facebook/dinov3-vitl16-pretrain-sat493m


config.json:   0%|          | 0.00/745 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 1.21GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/415 [00:00<?, ?it/s]

patch_size=16 hidden_size=1024 num_register_tokens=4
crop 224x224 -> grid 14x14 = 196 tokens


In [9]:
@torch.no_grad()
def extract_crop_features_batch(crops_uint8_bhw3):
    """crops (B, H, W, 3) uint8 -> (B, N_TOKENS, HIDDEN_SIZE) float32 numpy.
    Descarta CLS + register tokens ANTES do reshape."""
    t = torch.from_numpy(crops_uint8_bhw3.astype(np.float32) / 255.0).permute(0, 3, 1, 2)
    t = t.to(DEVICE)
    t = (t - mean_t) / std_t
    kwargs = {'interpolate_pos_encoding': True} if SUPPORTS_INTERP else {}
    out = model(pixel_values=t, **kwargs)
    tokens = out.last_hidden_state.float().cpu().numpy()
    patch_tokens = tokens[:, 1 + NUM_REGISTER_TOKENS:]
    if patch_tokens.shape[1] != N_TOKENS:
        raise RuntimeError(f'Esperava {N_TOKENS} patch tokens, vieram {patch_tokens.shape[1]}.')
    return patch_tokens

## 6. Coleta de crops por site (positivos + negativos, com checkpoint)

**Positivos**: 1 crop centrado em cada polígono, com label deduzido do nome do arquivo geojson (folha_larga=1, folha_estreita=2). Geojsons de mamona são carregados **só pra filtrar negativos** (não viram positivos).

**Negativos**: amostra `MAX_CULTIVO_CROPS_PER_SITE` pontos aleatórios da plantação (respeitando `INVERTED_PLANTACAO` pra Flaviano), rejeitando qualquer overlap com daninha (inclui mamona, pra não confundir o cultivo).

In [10]:
import re


def _norm(name):
    return re.sub(r'[^a-z0-9]', '', name.lower())


def class_from_geojson_path(path):
    """Retorna class_id (1..N-1) ou None se o geojson deve ser ignorado como positivo
    (mas ainda contar como daninha pra filtrar negativos)."""
    base = _norm(os.path.basename(path))
    for substr in IGNORE_GEOJSON_SUBSTR:
        if _norm(substr) in base:
            return None
    for substr, cls in GEOJSON_CLASS_MAP:
        if _norm(substr) in base:
            return cls
    raise ValueError(f'Nao consegui deduzir classe do arquivo: {path}')


def load_polys(path, raster_crs):
    if not os.path.exists(path):
        print(f'  aviso: arquivo nao encontrado, pulando: {path}')
        return []
    try:
        gdf = gpd.read_file(path)
    except Exception as e:
        print(f'  aviso: nao consegui ler {path}: {e}')
        return []
    if gdf.crs is not None and gdf.crs != raster_crs:
        gdf = gdf.to_crs(raster_crs)
    polys = []
    for i, g in enumerate(gdf.geometry):
        if g is None or g.is_empty:
            continue
        if not g.is_valid:
            g = g.buffer(0)
        if g.is_empty:
            continue
        polys.append((i, g))
    return polys


def read_centered_crop(src, center_col, center_row):
    half = CROP_PIXELS // 2
    col_off = int(round(center_col - half))
    row_off = int(round(center_row - half))
    win = Window(col_off, row_off, CROP_PIXELS, CROP_PIXELS)
    tile = src.read([1, 2, 3], window=win, boundless=True, fill_value=0)
    return np.moveaxis(tile, 0, -1)


def sample_cultivo_pixel_centers(src, plant_union, daninha_sindex, daninha_geoms,
                                 n_target, invert_plantacao, rng):
    if plant_union is None or plant_union.is_empty:
        return []
    minx, miny, maxx, maxy = plant_union.bounds
    if invert_plantacao:
        minx, miny = src.transform * (0, src.height)
        maxx, maxy = src.transform * (src.width, 0)
    centers = []
    max_tries = n_target * 15
    tries = 0
    while len(centers) < n_target and tries < max_tries:
        tries += 1
        x = rng.uniform(minx, maxx)
        y = rng.uniform(miny, maxy)
        pt = Point(x, y)
        if invert_plantacao:
            if plant_union.contains(pt):
                continue
        else:
            if not plant_union.contains(pt):
                continue
        if daninha_sindex is not None:
            hits = list(daninha_sindex.query(pt, predicate='intersects'))
            if hits:
                continue
        col, row = ~src.transform * (x, y)
        half = CROP_PIXELS // 2
        if col < half or col > src.width - half or row < half or row > src.height - half:
            continue
        centers.append((float(col), float(row), float(x), float(y)))
    if len(centers) < n_target:
        print(f'  aviso: amostrei {len(centers)}/{n_target} cultivo apos {tries} tentativas')
    return centers


def collect_site_crops(cfg):
    print(f"\n=== {cfg['nome']} ===")
    invert_plantacao = cfg['nome'] in INVERTED_PLANTACAO
    if invert_plantacao:
        print(f'  aviso: {cfg["nome"]} em INVERTED_PLANTACAO -- area valida = FORA do poligono')
    rng = np.random.default_rng(RANDOM_STATE)
    crops_out = []

    with rasterio.open(cfg['imagem']) as src:
        raster_crs = src.crs
        raster_w, raster_h = src.width, src.height

        # 1. carregar TODAS as daninhas (positivas e ignoradas) pra filtrar negativos
        all_daninha_geoms = []           # todas -- pra sindex do cultivo
        positive_items = []              # so as que viram positivo (com classe)
        for gj in cfg['geojsons']:
            cls = class_from_geojson_path(gj)   # None -> ignorada
            polys = load_polys(gj, raster_crs)
            for poly_id, geom in polys:
                all_daninha_geoms.append(geom)
                if cls is not None:
                    positive_items.append((cls, os.path.basename(gj), poly_id, geom))
            tag = f'classe={CLASS_NAMES[cls]}' if cls is not None else 'IGNORADA (so filtra negativos)'
            print(f'  {os.path.basename(gj)} {tag}: {len(polys)} poligonos')
        print(f'  total daninhas (todas): {len(all_daninha_geoms)} | positivos com classe: {len(positive_items)}')

        # 2. positivos: 1 crop por poligono anotado (com classe)
        pending_pos = []
        for cls, gj_src, poly_id, geom in positive_items:
            cx, cy = geom.centroid.x, geom.centroid.y
            col, row = ~src.transform * (cx, cy)
            if not (0 <= col < raster_w and 0 <= row < raster_h):
                continue
            minx, miny, maxx, maxy = geom.bounds
            pix_w = abs((maxx - minx) / src.transform.a)
            pix_h = abs((maxy - miny) / src.transform.e)
            if pix_w * pix_h < MIN_POLYGON_PIXELS:
                continue
            pending_pos.append((cls, gj_src, poly_id, float(col), float(row), float(cx), float(cy)))
        print(f'  positivos apos filtros: {len(pending_pos)}')

        # 3. negativos: amostra da plantacao evitando TODAS as daninhas (inclui mamona)
        plantacao_polys = load_polys(cfg['plantacao'], raster_crs) if cfg.get('plantacao') else []
        plant_union = unary_union([g.buffer(0) for _, g in plantacao_polys]) if plantacao_polys else None
        daninha_gs = gpd.GeoSeries(all_daninha_geoms) if all_daninha_geoms else None
        daninha_sindex = daninha_gs.sindex if daninha_gs is not None else None
        neg_centers = sample_cultivo_pixel_centers(
            src, plant_union, daninha_sindex, daninha_gs,
            MAX_CULTIVO_CROPS_PER_SITE, invert_plantacao, rng)
        print(f'  negativos amostrados: {len(neg_centers)}')

        # 4. processa em batches pela GPU
        BATCH_GPU = 32
        for i in tqdm(range(0, len(pending_pos), BATCH_GPU), desc=f'{cfg["nome"]} pos'):
            batch = pending_pos[i:i + BATCH_GPU]
            crops = np.stack([read_centered_crop(src, item[3], item[4]) for item in batch], axis=0)
            feats = extract_crop_features_batch(crops)
            for j, item in enumerate(batch):
                cls, gj_src, poly_id, col, row, lon, lat = item
                crops_out.append({
                    'features': feats[j].astype(np.float16),
                    'label': cls,
                    'meta': {
                        'site': cfg['nome'],
                        'kind': 'positive',
                        'geojson_source': gj_src,
                        'polygon_id': poly_id,
                        'centroid_col': col, 'centroid_row': row,
                        'centroid_lon': lon, 'centroid_lat': lat,
                    },
                })
            if DEVICE == 'cuda':
                torch.cuda.empty_cache()
        for i in tqdm(range(0, len(neg_centers), BATCH_GPU), desc=f'{cfg["nome"]} neg'):
            batch = neg_centers[i:i + BATCH_GPU]
            crops = np.stack([read_centered_crop(src, item[0], item[1]) for item in batch], axis=0)
            feats = extract_crop_features_batch(crops)
            for j, (col, row, lon, lat) in enumerate(batch):
                crops_out.append({
                    'features': feats[j].astype(np.float16),
                    'label': 0,
                    'meta': {
                        'site': cfg['nome'],
                        'kind': 'negative',
                        'geojson_source': '',
                        'polygon_id': -1,
                        'centroid_col': col, 'centroid_row': row,
                        'centroid_lon': lon, 'centroid_lat': lat,
                    },
                })
            if DEVICE == 'cuda':
                torch.cuda.empty_cache()

    per_class = {c: 0 for c in CLASS_NAMES}
    for c in crops_out:
        per_class[c['label']] += 1
    print(f'  crops finais: ' + ', '.join(f'{CLASS_NAMES[k]}={v}' for k, v in per_class.items()))
    return crops_out


def config_signature_site(cfg):
    geojson_stats = []
    all_files = list(cfg['geojsons']) + ([cfg['plantacao']] if cfg.get('plantacao') else [])
    for gj in all_files:
        if os.path.exists(gj):
            st = os.stat(gj)
            geojson_stats.append((os.path.basename(gj), st.st_size, int(st.st_mtime)))
        else:
            geojson_stats.append((os.path.basename(gj), 0, 0))
    payload = repr((
        'dinov3-3class', DINO_MODEL, CROP_PIXELS, PATCH, DINOV3_MEAN, DINOV3_STD,
        MIN_POLYGON_PIXELS, MAX_CULTIVO_CROPS_PER_SITE, RANDOM_STATE,
        GEOJSON_CLASS_MAP, IGNORE_GEOJSON_SUBSTR,
        cfg['nome'], geojson_stats,
    ))
    return hashlib.sha256(payload.encode()).hexdigest()[:16]


def collect_site_crops_cached(cfg):
    sig = config_signature_site(cfg)
    ck_path = f"{CHECKPOINT_DIR}/{cfg['nome']}_{sig}.joblib"
    if os.path.exists(ck_path):
        d = joblib.load(ck_path)
        per_class = {c: 0 for c in CLASS_NAMES}
        for c in d:
            per_class[c['label']] += 1
        print(f"  (cache: {cfg['nome']} -- " + ', '.join(f'{CLASS_NAMES[k]}={v}' for k, v in per_class.items()) + ')')
        return d
    crops = collect_site_crops(cfg)
    joblib.dump(crops, ck_path)
    return crops

## 7. Rodar coleta pra todos os sites

In [11]:
all_crops = []
for cfg in PARES_CONFIG:
    all_crops.extend(collect_site_crops_cached(cfg))

per_class = {c: 0 for c in CLASS_NAMES}
for c in all_crops:
    per_class[c['label']] += 1
print(f'\nTotal: {len(all_crops)} crops | ' + ', '.join(f'{CLASS_NAMES[k]}={v}' for k, v in per_class.items()))
assert min(per_class.values()) > 0, 'Preciso de crops em TODAS as classes.'

  (cache: Giasa -- cultivo=1500, folha_larga=1461, folha_estreita=1286)
  (cache: DoisRiosFlaviano -- cultivo=1500, folha_larga=1602, folha_estreita=1250)
  (cache: Flaviano -- cultivo=1500, folha_larga=296, folha_estreita=228)
  (cache: CelsoSTE2 -- cultivo=1500, folha_larga=116, folha_estreita=158)
  (cache: Celso01 -- cultivo=1500, folha_larga=171, folha_estreita=7)

Total: 14075 crops | cultivo=7500, folha_larga=3646, folha_estreita=2929


## 8. Split treino/validação (faixa espacial dentro de cada fazenda)

Split respeita TODAS as classes: pra cada site, ordena todos os crops pelo centroide em pixels no eixo `SPLIT_AXIS`. 20% mais à direita = val, gap de `MARGIN_TILES * CROP_PIXELS` px.

In [12]:
def split_train_val(all_crops, val_fraction, margin_tiles, axis='x'):
    if axis not in ('x', 'y'):
        raise ValueError(f"SPLIT_AXIS invalido: {axis!r}")
    key = 'centroid_col' if axis == 'x' else 'centroid_row'
    margin_px = margin_tiles * CROP_PIXELS

    train, val = [], []
    by_site = {}
    for c in all_crops:
        by_site.setdefault(c['meta']['site'], []).append(c)

    for site, crops in by_site.items():
        crops_sorted = sorted(crops, key=lambda c: c['meta'][key])
        total = len(crops_sorted)
        if total == 0:
            continue
        n_val_target = int(round(total * val_fraction))
        if n_val_target == 0:
            train.extend(crops_sorted)
            print(f'  {site}: total={total} val=0 (fraction {val_fraction} muito pequena)')
            continue
        val_start_idx = total - n_val_target
        val_start_pos = crops_sorted[val_start_idx]['meta'][key]
        train_end_pos = val_start_pos - margin_px

        n_tr = n_va = n_gap = 0
        n_tr_by = {c: 0 for c in CLASS_NAMES}
        n_va_by = {c: 0 for c in CLASS_NAMES}
        for c in crops_sorted:
            p = c['meta'][key]
            if p >= val_start_pos:
                val.append(c); n_va += 1
                n_va_by[c['label']] += 1
            elif p < train_end_pos:
                train.append(c); n_tr += 1
                n_tr_by[c['label']] += 1
            else:
                n_gap += 1
        tr_str = ','.join(f'{CLASS_NAMES[k]}={v}' for k, v in n_tr_by.items())
        va_str = ','.join(f'{CLASS_NAMES[k]}={v}' for k, v in n_va_by.items())
        print(f'  {site}: treino={n_tr} ({tr_str}) | val={n_va} ({va_str}) | gap={n_gap}')
    return train, val


train_crops, val_crops = split_train_val(all_crops, VAL_FRACTION, MARGIN_TILES, SPLIT_AXIS)
print(f'\nTotal treino: {len(train_crops)} | val: {len(val_crops)}')

# Pesos de classe
if CLASS_WEIGHTS is None:
    counts = np.zeros(N_CLASSES, dtype=np.float64)
    for c in train_crops:
        counts[c['label']] += 1
    counts_safe = np.maximum(counts, 1.0)
    weights = counts.sum() / (N_CLASSES * counts_safe)
    weights = np.clip(weights, 0.1, 20.0)
    class_weights = weights.astype(np.float32)
else:
    class_weights = np.array(CLASS_WEIGHTS, dtype=np.float32)
print('Pesos por classe: ' + ', '.join(f'{CLASS_NAMES[i]}={class_weights[i]:.2f}' for i in range(N_CLASSES)))

  Giasa: treino=3357 (cultivo=1238,folha_larga=959,folha_estreita=1160) | val=849 (cultivo=250,folha_larga=479,folha_estreita=120) | gap=41
  DoisRiosFlaviano: treino=3459 (cultivo=1261,folha_larga=1284,folha_estreita=914) | val=870 (cultivo=233,folha_larga=310,folha_estreita=327) | gap=23
  Flaviano: treino=1577 (cultivo=1167,folha_larga=203,folha_estreita=207) | val=405 (cultivo=306,folha_larga=83,folha_estreita=16) | gap=42
  CelsoSTE2: treino=1397 (cultivo=1211,folha_larga=87,folha_estreita=99) | val=355 (cultivo=268,folha_larga=28,folha_estreita=59) | gap=22
  Celso01: treino=1335 (cultivo=1203,folha_larga=128,folha_estreita=4) | val=336 (cultivo=291,folha_larga=42,folha_estreita=3) | gap=7

Total treino: 11125 | val: 2815
Pesos por classe: cultivo=0.61, folha_larga=1.39, folha_estreita=1.56


## 9. Dataset e DataLoader

In [13]:
class CropFeaturesDataset(Dataset):
    def __init__(self, crops):
        self.crops = crops

    def __len__(self):
        return len(self.crops)

    def __getitem__(self, idx):
        c = self.crops[idx]
        feats = torch.from_numpy(c['features'].astype(np.float32))
        label = torch.tensor(c['label'], dtype=torch.long)
        return feats, label


train_loader = DataLoader(CropFeaturesDataset(train_crops), batch_size=BATCH, shuffle=True, drop_last=False)
val_loader = None
if len(val_crops) > 0:
    val_loader = DataLoader(CropFeaturesDataset(val_crops), batch_size=BATCH, shuffle=False, drop_last=False)
print(f'batches treino: {len(train_loader)}' + (f' | val: {len(val_loader)}' if val_loader else ''))

batches treino: 174 | val: 44


## 10. Head Transformer + mean pool → Linear(1024, N_CLASSES)

Mesma receita do notebook base (LayerNorm + pos_encoding + 2× TransformerEncoderLayer GELU norm_first), com mean pool no fim pra colapsar os 196 tokens em 1 vetor por crop.

In [14]:
class CropTransformerHead(nn.Module):
    def __init__(self, hidden_size, n_tokens, n_layers=2, nhead=8, dropout=0.1,
                 classifier_dropout=0.0, n_classes=2):
        super().__init__()
        self.input_norm = nn.LayerNorm(hidden_size)
        self.pos_encoding = nn.Parameter(torch.zeros(1, n_tokens, hidden_size))
        nn.init.trunc_normal_(self.pos_encoding, std=0.02)
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=hidden_size, nhead=nhead, dim_feedforward=hidden_size * 2,
            dropout=dropout, batch_first=True, norm_first=True, activation='gelu',
        )
        self.encoder = nn.TransformerEncoder(encoder_layer, num_layers=n_layers)
        self.norm = nn.LayerNorm(hidden_size)
        self.class_dropout = nn.Dropout(classifier_dropout)
        self.classifier = nn.Linear(hidden_size, n_classes)

    def forward(self, tokens):
        x = self.input_norm(tokens) + self.pos_encoding
        x = self.encoder(x)
        x = self.norm(x)
        x = x.mean(dim=1)
        x = self.class_dropout(x)
        return self.classifier(x)


head = CropTransformerHead(
    hidden_size=HIDDEN_SIZE, n_tokens=N_TOKENS, n_layers=TRANSFORMER_LAYERS,
    nhead=NHEAD, dropout=TRANSFORMER_DROPOUT, classifier_dropout=CLASSIFIER_DROPOUT,
    n_classes=N_CLASSES,
).to(DEVICE)
n_params = sum(p.numel() for p in head.parameters() if p.requires_grad)
print(f'Head transformer: {n_params:,} parametros treinaveis')

Head transformer: 17,007,619 parametros treinaveis


/tmp/ipykernel_11568/1517265315.py:12: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.encoder = nn.TransformerEncoder(encoder_layer, num_layers=n_layers)


## 11. Loss, otimizador, scheduler

In [15]:
class_weight_t = torch.tensor(class_weights, dtype=torch.float32, device=DEVICE)
loss_fn = nn.CrossEntropyLoss(weight=class_weight_t, label_smoothing=LABEL_SMOOTHING)

optimizer = torch.optim.AdamW(head.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)

def lr_lambda(epoch_idx):
    if WARMUP_EPOCHS > 0 and epoch_idx < WARMUP_EPOCHS:
        return (epoch_idx + 1) / WARMUP_EPOCHS
    denom = max(EPOCHS - WARMUP_EPOCHS, 1)
    progress = (epoch_idx - WARMUP_EPOCHS) / denom
    progress = min(max(progress, 0.0), 1.0)
    return 0.5 * (1.0 + math.cos(math.pi * progress))

scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda=lr_lambda)


def run_epoch(loader, train=True):
    head.train(mode=train)
    total_loss, n_batches = 0.0, 0
    for tokens, labels in loader:
        tokens = tokens.to(DEVICE, non_blocking=True)
        labels = labels.to(DEVICE, non_blocking=True)
        with torch.set_grad_enabled(train):
            logits = head(tokens)
            loss = loss_fn(logits, labels)
        if train:
            optimizer.zero_grad()
            loss.backward()
            nn.utils.clip_grad_norm_(head.parameters(), GRAD_CLIP_NORM)
            optimizer.step()
        total_loss += float(loss.item())
        n_batches += 1
    return total_loss / max(n_batches, 1)


@torch.no_grad()
def evaluate(loader):
    """Retorna (macro_f1, y_true, y_pred). macro_f1 usado pro early stop."""
    if loader is None:
        return None, None, None
    head.eval()
    all_true, all_pred = [], []
    for tokens, labels in loader:
        tokens = tokens.to(DEVICE, non_blocking=True)
        logits = head(tokens)
        pred = logits.argmax(dim=-1).cpu().numpy()
        all_true.append(labels.numpy())
        all_pred.append(pred)
    y_true = np.concatenate(all_true)
    y_pred = np.concatenate(all_pred)
    macro_f1 = f1_score(y_true, y_pred, average='macro', zero_division=0)
    return macro_f1, y_true, y_pred


## 12. Loop de treino (early stop em macro-F1)

Macro-F1 (média das F1s por classe, sem peso) — trata cultivo/folha_larga/folha_estreita como igualmente importantes. Checkpoint por época em `train_ckpt_c{N}.pt` sobrevive a disconnect do Colab.

In [16]:
# ---- checkpoint por epoca (retomada apos disconnect) ----
RESUME_CKPT = f'/content/drive/MyDrive/rigtech_ciclos/train_ckpt_c{CICLO}.pt'

best_f1 = -1.0
best_state = None
epochs_no_improve = 0
start_epoch = 1

if os.path.exists(RESUME_CKPT):
    ck = torch.load(RESUME_CKPT, map_location=DEVICE)
    head.load_state_dict(ck['head_state_dict'])
    optimizer.load_state_dict(ck['optimizer_state_dict'])
    scheduler.load_state_dict(ck['scheduler_state_dict'])
    start_epoch = ck['epoch'] + 1
    best_f1 = ck['best_f1']
    best_state = ck.get('best_state')
    epochs_no_improve = ck['epochs_no_improve']
    print(f'retomando treino da epoca {start_epoch} (melhor F1 ate agora: {best_f1:.4f})')

for epoch in range(start_epoch, EPOCHS + 1):
    train_loss = run_epoch(train_loader, train=True)
    val_f1, _, _ = evaluate(val_loader)
    current_lr = optimizer.param_groups[0]['lr']
    msg = f'epoca {epoch}/{EPOCHS} | LR: {current_lr:.2e} | loss treino: {train_loss:.4f}'
    if val_f1 is not None:
        msg += f' | macro-F1 (val): {val_f1:.4f}'
        if val_f1 > best_f1:
            best_f1 = val_f1
            best_state = {k: v.detach().cpu().clone() for k, v in head.state_dict().items()}
            epochs_no_improve = 0
            msg += '  (melhor, salvando)'
        else:
            epochs_no_improve += 1
    else:
        best_state = {k: v.detach().cpu().clone() for k, v in head.state_dict().items()}
    print(msg)
    scheduler.step()

    # salva checkpoint desta epoca (sobrescreve o anterior)
    torch.save({
        'epoch': epoch,
        'head_state_dict': head.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
        'scheduler_state_dict': scheduler.state_dict(),
        'best_f1': best_f1,
        'best_state': best_state,
        'epochs_no_improve': epochs_no_improve,
    }, RESUME_CKPT)

    if val_loader is not None and epochs_no_improve >= EARLY_STOP_PATIENCE:
        print(f'early stop -- {EARLY_STOP_PATIENCE} epocas sem melhora.')
        break

if best_state is not None:
    head.load_state_dict(best_state)
print('Treino concluido.' + (f' Melhor macro-F1 (val): {best_f1:.4f}' if best_f1 >= 0 else ''))

torch.save({
    'head_state_dict': head.state_dict(),
    'hidden_size': HIDDEN_SIZE, 'n_tokens': N_TOKENS, 'n_classes': N_CLASSES,
    'transformer_layers': TRANSFORMER_LAYERS, 'nhead': NHEAD,
    'transformer_dropout': TRANSFORMER_DROPOUT, 'classifier_dropout': CLASSIFIER_DROPOUT,
    'label_smoothing': LABEL_SMOOTHING, 'class_weights': class_weights.tolist(),
    'dino_model': DINO_MODEL, 'dinov3_mean': DINOV3_MEAN, 'dinov3_std': DINOV3_STD,
    'crop_pixels': CROP_PIXELS, 'patch': PATCH, 'num_register_tokens': NUM_REGISTER_TOKENS,
    'class_names': CLASS_NAMES, 'best_val_macro_f1': best_f1, 'ciclo': CICLO,
}, OUTPUT_MODEL)
print(f'Modelo salvo em {OUTPUT_MODEL}')

# treino terminou -- limpa checkpoint de retomada (nao serve mais)
if os.path.exists(RESUME_CKPT):
    os.remove(RESUME_CKPT)
    print(f'checkpoint de retomada removido: {RESUME_CKPT}')

epoca 1/25 | LR: 5.00e-05 | loss treino: 0.8093 | macro-F1 (val): 0.6914  (melhor, salvando)
epoca 2/25 | LR: 1.00e-04 | loss treino: 0.6511 | macro-F1 (val): 0.7290  (melhor, salvando)
epoca 3/25 | LR: 1.00e-04 | loss treino: 0.5694 | macro-F1 (val): 0.7374  (melhor, salvando)
epoca 4/25 | LR: 9.95e-05 | loss treino: 0.5227 | macro-F1 (val): 0.7756  (melhor, salvando)
epoca 5/25 | LR: 9.81e-05 | loss treino: 0.4682 | macro-F1 (val): 0.7750
epoca 6/25 | LR: 9.59e-05 | loss treino: 0.4269 | macro-F1 (val): 0.7625
epoca 7/25 | LR: 9.27e-05 | loss treino: 0.3748 | macro-F1 (val): 0.7695
epoca 8/25 | LR: 8.88e-05 | loss treino: 0.3310 | macro-F1 (val): 0.7786  (melhor, salvando)
epoca 9/25 | LR: 8.41e-05 | loss treino: 0.2947 | macro-F1 (val): 0.7509
epoca 10/25 | LR: 7.88e-05 | loss treino: 0.2695 | macro-F1 (val): 0.7751
epoca 11/25 | LR: 7.30e-05 | loss treino: 0.2445 | macro-F1 (val): 0.7749
epoca 12/25 | LR: 6.67e-05 | loss treino: 0.2258 | macro-F1 (val): 0.7631
epoca 13/25 | LR: 6.0

## 13. Relatório de métricas

In [17]:
report_lines = [f'=== ciclo {CICLO} | DINOv3+Transformer per-crop 3-class ===']
report_lines.append(f'Classes: ' + ', '.join(f'{k}={v}' for k, v in CLASS_NAMES.items()))
report_lines.append(f'Split: {VAL_FRACTION:.0%} da faixa "{SPLIT_AXIS}" por site, margem={MARGIN_TILES} tiles ({MARGIN_TILES*CROP_PIXELS}px)')
per_class_all = {c: 0 for c in CLASS_NAMES}
for c in all_crops:
    per_class_all[c['label']] += 1
report_lines.append(f'Total crops: {len(all_crops)} (' + ', '.join(f'{CLASS_NAMES[k]}={v}' for k, v in per_class_all.items()) + ')')
report_lines.append(f'Treino: {len(train_crops)} | Val: {len(val_crops)}')
report_lines.append('')

if val_loader is not None:
    macro_f1, y_true, y_pred = evaluate(val_loader)
    accuracy = (y_true == y_pred).mean()
    report_lines.append(f'macro-F1 (val): {macro_f1:.4f}')
    report_lines.append(f'accuracy (val): {accuracy:.4f}')
    report_lines.append('')
    target_names = [CLASS_NAMES[i] for i in range(N_CLASSES)]
    report_lines.append(classification_report(y_true, y_pred, target_names=target_names, zero_division=0))
    report_lines.append('Matriz de confusao [linhas=verdadeiro, colunas=previsto]:')
    report_lines.append('              ' + '  '.join(f'{CLASS_NAMES[i]:>16}' for i in range(N_CLASSES)))
    cm = confusion_matrix(y_true, y_pred, labels=list(range(N_CLASSES)))
    for i, row in enumerate(cm):
        report_lines.append(f'{CLASS_NAMES[i]:>14} ' + '  '.join(f'{v:>16d}' for v in row))
else:
    report_lines.append('sem tiles de validacao -- relatorio de metricas pulado.')

report_text = '\n'.join(report_lines)
print(report_text)
with open(REPORT_PATH, 'w', encoding='utf-8') as f:
    f.write(report_text)
print(f'\nRelatorio salvo em {REPORT_PATH}')

=== ciclo 1 | DINOv3+Transformer per-crop 3-class ===
Classes: 0=cultivo, 1=folha_larga, 2=folha_estreita
Split: 20% da faixa "x" por site, margem=1 tiles (224px)
Total crops: 14075 (cultivo=7500, folha_larga=3646, folha_estreita=2929)
Treino: 11125 | Val: 2815

macro-F1 (val): 0.7786
accuracy (val): 0.8153

                precision    recall  f1-score   support

       cultivo       0.91      0.88      0.90      1348
   folha_larga       0.77      0.83      0.80       942
folha_estreita       0.66      0.63      0.64       525

      accuracy                           0.82      2815
     macro avg       0.78      0.78      0.78      2815
  weighted avg       0.82      0.82      0.82      2815

Matriz de confusao [linhas=verdadeiro, colunas=previsto]:
                       cultivo       folha_larga    folha_estreita
       cultivo             1186                88                74
   folha_larga               64               778               100
folha_estreita               51   

## 14. Exportar suspeitas → CSV pra revisão humana

**Só sobre POSITIVOS anotados** (folha_larga e folha_estreita). Negativos (cultivo) são amostrados aleatoriamente, então erro do modelo neles não indica anotação errada.

- **`misclass`**: modelo prediz classe **diferente** da anotada (ex: rotulado folha_larga mas modelo diz folha_estreita ou cultivo). Ordenado por confidence decrescente na classe predita (mais confiante = erro mais provável).
- **`low_conf`**: modelo acerta a classe mas com `prob_da_classe_verdadeira < SUSPECT_LOW_CONF_THRESHOLD` → geometria ambígua.

In [18]:
@torch.no_grad()
def infer_all(crops):
    head.eval()
    loader = DataLoader(CropFeaturesDataset(crops), batch_size=BATCH, shuffle=False)
    preds, probs_all = [], []
    for tokens, _ in loader:
        tokens = tokens.to(DEVICE, non_blocking=True)
        logits = head(tokens)
        probs = torch.softmax(logits, dim=-1).cpu().numpy()
        preds.append(probs.argmax(axis=-1))
        probs_all.append(probs)
    return np.concatenate(preds), np.concatenate(probs_all, axis=0)


# so os positivos anotados (folha_larga e folha_estreita)
positive_crops = [c for c in all_crops if c['meta']['kind'] == 'positive']
print(f'Analisando {len(positive_crops)} positivos anotados...')

pos_preds, pos_probs = infer_all(positive_crops)

suspects = []
for i, c in enumerate(positive_crops):
    true = int(c['label'])
    pred = int(pos_preds[i])
    prob_true = float(pos_probs[i, true])
    prob_pred = float(pos_probs[i, pred])
    if pred != true:
        kind = 'misclass'
        sort_confidence = prob_pred
    elif prob_true < SUSPECT_LOW_CONF_THRESHOLD:
        kind = 'low_conf'
        sort_confidence = prob_true
    else:
        continue
    entry = {
        **c['meta'],
        'true_class': CLASS_NAMES[true],
        'pred_class': CLASS_NAMES[pred],
        'prob_true_class': prob_true,
        'prob_pred_class': prob_pred,
        'kind': kind,
    }
    for k in range(N_CLASSES):
        entry[f'prob_{CLASS_NAMES[k]}'] = float(pos_probs[i, k])
    suspects.append(entry)

# misclass primeiro (por prob_pred_class decrescente = mais confiante que e' a errada),
# depois low_conf (por prob_true_class crescente = mais duvidoso)
suspects.sort(key=lambda s: (0 if s['kind'] == 'misclass' else 1,
                             -s['prob_pred_class'] if s['kind'] == 'misclass' else s['prob_true_class']))

n_mis = sum(1 for s in suspects if s['kind'] == 'misclass')
n_lc = sum(1 for s in suspects if s['kind'] == 'low_conf')
print(f'Suspeitas: {len(suspects)} ({n_mis} misclass, {n_lc} low_conf) de {len(positive_crops)} positivos')

if suspects:
    prob_cols = [f'prob_{CLASS_NAMES[k]}' for k in range(N_CLASSES)]
    fieldnames = ['kind', 'site', 'geojson_source', 'polygon_id',
                  'true_class', 'pred_class', 'prob_true_class', 'prob_pred_class',
                  *prob_cols,
                  'centroid_lon', 'centroid_lat', 'centroid_col', 'centroid_row']
    with open(SUSPECTS_CSV, 'w', newline='', encoding='utf-8') as f:
        w = csv.DictWriter(f, fieldnames=fieldnames)
        w.writeheader()
        for s in suspects:
            w.writerow({k: s.get(k, '') for k in fieldnames})
    print(f'Suspeitas salvas em {SUSPECTS_CSV}')
    print('\nTop 10 suspeitas:')
    for s in suspects[:10]:
        print(f"  [{s['kind']}] {s['site']}/{s['geojson_source']}#{s['polygon_id']} "
              f"true={s['true_class']} pred={s['pred_class']} "
              f"prob_pred={s['prob_pred_class']:.3f} prob_true={s['prob_true_class']:.3f}")

Analisando 6575 positivos anotados...
Suspeitas: 655 (534 misclass, 121 low_conf) de 6575 positivos
Suspeitas salvas em /content/drive/MyDrive/suspeitas_c1.csv

Top 10 suspeitas:
  [misclass] DoisRiosFlaviano/FolhaLargaDoisRiosFlaviano.geojson#923 true=folha_larga pred=folha_estreita prob_pred=0.995 prob_true=0.003
  [misclass] CelsoSTE2/FolhaEstreitaCelsoSTE-2 (1).geojson#65 true=folha_estreita pred=folha_larga prob_pred=0.994 prob_true=0.002
  [misclass] DoisRiosFlaviano/FolhaEstreitaDoisRiosFlaviano.geojson#782 true=folha_estreita pred=folha_larga prob_pred=0.994 prob_true=0.002
  [misclass] DoisRiosFlaviano/FolhaLargaDoisRiosFlaviano.geojson#829 true=folha_larga pred=folha_estreita prob_pred=0.994 prob_true=0.003
  [misclass] Giasa/FolhaEstreitaGiasa (1).geojson#1194 true=folha_estreita pred=folha_larga prob_pred=0.992 prob_true=0.003
  [misclass] Giasa/FolhaEstreitaGiasa (1).geojson#1178 true=folha_estreita pred=folha_larga prob_pred=0.992 prob_true=0.003
  [misclass] Giasa/FolhaE

In [2]:
import os
import csv
import requests
from getpass import getpass
from collections import Counter

CICLO = 1
LINEAR_TEAM_KEY = 'RIG'
LINEAR_ENDPOINT = 'https://api.linear.app/graphql'
LINEAR_TOP_N_PER_CARD = 20
LINEAR_STATE_NAME = 'Selecionadas'
SUSPECTS_CSV = f'/content/drive/MyDrive/suspeitas_c{CICLO}.csv'

LINEAR_API_KEY = os.environ.get('LINEAR_API_KEY') or getpass('LINEAR_API_KEY: ')
assert LINEAR_API_KEY

# Verifica se o arquivo CSV de suspeitas existe antes de tentar abri-lo
if not os.path.exists(SUSPECTS_CSV):
    raise FileNotFoundError(
        f"O arquivo {SUSPECTS_CSV} não foi encontrado. "
        "Por favor, verifique se a célula anterior que gera o CSV "
        "('14. Exportar suspeitas → CSV pra revisão humana') foi executada com sucesso "
        "e se o Google Drive está montado e acessível. "
        "Pode ser necessário executar a célula anterior novamente ou remontar o Drive."
    )

suspects = []
with open(SUSPECTS_CSV) as f:
    for row in csv.DictReader(f):
        for k in ('prob_true_class', 'prob_pred_class', 'centroid_lon', 'centroid_lat'):
            row[k] = float(row[k])
        suspects.append(row)
print(len(suspects), 'suspeitas carregadas do CSV')


def gql(query, variables=None):
    headers = {'Authorization': LINEAR_API_KEY, 'Content-Type': 'application/json'}
    payload = {'query': query, 'variables': variables or {}}
    r = requests.post(LINEAR_ENDPOINT, headers=headers, json=payload)
    r.raise_for_status()
    data = r.json()
    if 'errors' in data:
        raise RuntimeError(data['errors'])
    return data['data']


Q_TEAMS = """
query {
  teams { nodes { id key name } }
}
"""

Q_STATES = """
query($t: String!) {
  workflowStates(filter: {team: {id: {eq: $t}}}) { nodes { id name } }
}
"""

Q_LABELS = """
query($t: String!) {
  issueLabels(filter: {team: {id: {eq: $t}}}) { nodes { id name } }
}
"""

M_LABEL_CREATE = """
mutation($n: String!, $t: String!) {
  issueLabelCreate(input: {name: $n, teamId: $t}) { issueLabel { id } }
}
"""

M_ISSUE_CREATE = """
mutation($t: String!, $d: String!, $team: String!, $state: String!, $labels: [String!]) {
  issueCreate(input: {title: $t, description: $d, teamId: $team, stateId: $state, labelIds: $labels}) {
    success
    issue { id identifier url }
  }
}
"""

teams_data = gql(Q_TEAMS)
teams = teams_data['teams']['nodes']
team = next(t for t in teams if t['key'] == LINEAR_TEAM_KEY)
TEAM_ID = team['id']

states_data = gql(Q_STATES, {'t': TEAM_ID})
states = states_data['workflowStates']['nodes']
state = next((s for s in states if s['name'] == LINEAR_STATE_NAME), None)
STATE_ID = state['id']
print('time=', LINEAR_TEAM_KEY, 'state=', state['name'])

labels_data = gql(Q_LABELS, {'t': TEAM_ID})
labels = labels_data['issueLabels']['nodes']
label_by_name = {l['name'].lower(): l['id'] for l in labels}


def ensure_label(name):
    key = name.lower()
    if key in label_by_name:
        return label_by_name[key]
    resp = gql(M_LABEL_CREATE, {'n': name, 't': TEAM_ID})
    lid = resp['issueLabelCreate']['issueLabel']['id']
    label_by_name[key] = lid
    return lid


ciclo_lbl = ensure_label(f'ciclo-{CICLO}')
run_lbl = ensure_label('run-modelo')
susp_lbl = ensure_label('suspeita')


def create_issue(title, description, label_ids):
    variables = {
        't': title,
        'd': description,
        'team': TEAM_ID,
        'state': STATE_ID,
        'labels': label_ids,
    }
    resp = gql(M_ISSUE_CREATE, variables)
    return resp['issueCreate']['issue']


by_group = {}
for s in suspects:
    by_group.setdefault((s['site'], s['kind']), []).append(s)

created = []
groups_sorted = sorted(by_group.items(), key=lambda kv: -len(kv[1]))

for (site, kind), group in groups_sorted:
    direction = ''
    if kind == 'misclass':
        pair_counter = Counter((s['true_class'], s['pred_class']) for s in group)
        (t, p), n = pair_counter.most_common(1)[0]
        direction = f' ({t} -> {p}: {n}/{len(group)})'
    title = f'[c{CICLO}] {site} - {len(group)} {kind}{direction}'

    if kind == 'misclass':
        group_sorted = sorted(group, key=lambda s: -s['prob_pred_class'])
    else:
        group_sorted = sorted(group, key=lambda s: s['prob_true_class'])
    top = group_sorted[:LINEAR_TOP_N_PER_CARD]

    class_counter = Counter(s['true_class'] for s in group)
    class_summary = ', '.join(f'{c}={n}' for c, n in class_counter.items())

    lines = []
    lines.append(f'**Ciclo**: {CICLO}')
    lines.append(f'**Site**: {site}')
    lines.append(f'**Tipo**: `{kind}`')
    lines.append(f'**Total no grupo**: {len(group)} poligonos')
    lines.append(f'**Distribuicao de classes verdadeiras**: {class_summary}')
    lines.append('')
    lines.append(f'## Top {len(top)} poligonos pra revisar')
    lines.append('')
    lines.append('| # | geojson | poly_id | true | pred | prob_pred | prob_true | lon | lat |')
    lines.append('|---|---------|---------|------|------|-----------|-----------|-----|-----|')
    for i, s in enumerate(top, 1):
        row = (
            f"| {i} | `{s['geojson_source']}` | {s['polygon_id']} | "
            f"{s['true_class']} | **{s['pred_class']}** | "
            f"{s['prob_pred_class']:.3f} | {s['prob_true_class']:.3f} | "
            f"{s['centroid_lon']:.6f} | {s['centroid_lat']:.6f}"
        )
        lines.append(row)
    lines.append('')
    lines.append('## Como revisar')
    lines.append('1. Abrir o geojson correspondente no QGIS.')
    lines.append('2. Localizar poligono pelo `polygon_id` (ou pelas coordenadas).')
    lines.append('3. Decidir: **mover**, **redesenhar** ou **descartar**.')
    lines.append('4. Salvar geojson no Drive.')
    lines.append('5. Ao concluir o site, mover card pra `Aprovado`.')
    lines.append('')
    lines.append(f'CSV completo: `{SUSPECTS_CSV}` filtrado por `site={site}` e `kind={kind}`.')

    kind_lbl = ensure_label('misclass' if kind == 'misclass' else 'low-conf')
    description = '\n'.join(lines)
    issue = create_issue(title, description, [ciclo_lbl, run_lbl, susp_lbl, kind_lbl])
    print('criado:', issue['identifier'], issue['url'])
    created.append(issue)

print()
print('Total de cards criados:', len(created))

LINEAR_API_KEY: ··········
655 suspeitas carregadas do CSV


HTTPError: 400 Client Error: Bad Request for url: https://api.linear.app/graphql

In [3]:
import os
import csv
import requests
from getpass import getpass
from collections import Counter

CICLO = 1
LINEAR_TEAM_KEY = 'RIG'
LINEAR_ENDPOINT = 'https://api.linear.app/graphql'
LINEAR_TOP_N_PER_CARD = 20
LINEAR_STATE_NAME = 'Selecionadas'
SUSPECTS_CSV = f'/content/drive/MyDrive/suspeitas_c{CICLO}.csv'

LINEAR_API_KEY = os.environ.get('LINEAR_API_KEY') or getpass('LINEAR_API_KEY: ')
assert LINEAR_API_KEY

suspects = []
with open(SUSPECTS_CSV) as f:
    for row in csv.DictReader(f):
        for k in ('prob_true_class', 'prob_pred_class', 'centroid_lon', 'centroid_lat'):
            row[k] = float(row[k])
        suspects.append(row)
print(len(suspects), 'suspeitas carregadas do CSV')


def gql(query, variables=None):
    headers = {'Authorization': LINEAR_API_KEY, 'Content-Type': 'application/json'}
    payload = {'query': query, 'variables': variables or {}}
    r = requests.post(LINEAR_ENDPOINT, headers=headers, json=payload)
    data = r.json()
    if 'errors' in data:
        raise RuntimeError(data['errors'])
    r.raise_for_status()
    return data['data']


Q_TEAM_FULL = """
query($id: String!) {
  team(id: $id) {
    id
    key
    name
    states { nodes { id name } }
    labels { nodes { id name } }
  }
}
"""

Q_TEAMS = """
query {
  teams { nodes { id key name } }
}
"""

M_LABEL_CREATE = """
mutation($n: String!, $t: String!) {
  issueLabelCreate(input: {name: $n, teamId: $t}) { issueLabel { id name } }
}
"""

M_ISSUE_CREATE = """
mutation($t: String!, $d: String!, $team: String!, $state: String!, $labels: [String!]) {
  issueCreate(input: {title: $t, description: $d, teamId: $team, stateId: $state, labelIds: $labels}) {
    success
    issue { id identifier url }
  }
}
"""

# passo 1: acha o id do time pela key
teams = gql(Q_TEAMS)['teams']['nodes']
team_meta = next(t for t in teams if t['key'] == LINEAR_TEAM_KEY)
TEAM_ID = team_meta['id']

# passo 2: puxa states + labels via team direto (evita o filter quebrado)
team_full = gql(Q_TEAM_FULL, {'id': TEAM_ID})['team']
states = team_full['states']['nodes']
labels = team_full['labels']['nodes']

state = next((s for s in states if s['name'] == LINEAR_STATE_NAME), states[0])
STATE_ID = state['id']
print('time=', LINEAR_TEAM_KEY, 'state=', state['name'])

label_by_name = {l['name'].lower(): l['id'] for l in labels}


def ensure_label(name):
    key = name.lower()
    if key in label_by_name:
        return label_by_name[key]
    resp = gql(M_LABEL_CREATE, {'n': name, 't': TEAM_ID})
    lid = resp['issueLabelCreate']['issueLabel']['id']
    label_by_name[key] = lid
    return lid


ciclo_lbl = ensure_label(f'ciclo-{CICLO}')
run_lbl = ensure_label('run-modelo')
susp_lbl = ensure_label('suspeita')


def create_issue(title, description, label_ids):
    variables = {
        't': title,
        'd': description,
        'team': TEAM_ID,
        'state': STATE_ID,
        'labels': label_ids,
    }
    resp = gql(M_ISSUE_CREATE, variables)
    return resp['issueCreate']['issue']


by_group = {}
for s in suspects:
    by_group.setdefault((s['site'], s['kind']), []).append(s)

created = []
groups_sorted = sorted(by_group.items(), key=lambda kv: -len(kv[1]))

for (site, kind), group in groups_sorted:
    direction = ''
    if kind == 'misclass':
        pair_counter = Counter((s['true_class'], s['pred_class']) for s in group)
        (t, p), n = pair_counter.most_common(1)[0]
        direction = f' ({t} -> {p}: {n}/{len(group)})'
    title = f'[c{CICLO}] {site} - {len(group)} {kind}{direction}'

    if kind == 'misclass':
        group_sorted = sorted(group, key=lambda s: -s['prob_pred_class'])
    else:
        group_sorted = sorted(group, key=lambda s: s['prob_true_class'])
    top = group_sorted[:LINEAR_TOP_N_PER_CARD]

    class_counter = Counter(s['true_class'] for s in group)
    class_summary = ', '.join(f'{c}={n}' for c, n in class_counter.most_common())

    lines = []
    lines.append(f'**Ciclo**: {CICLO}')
    lines.append(f'**Site**: {site}')
    lines.append(f'**Tipo**: `{kind}`')
    lines.append(f'**Total no grupo**: {len(group)} poligonos')
    lines.append(f'**Distribuicao de classes verdadeiras**: {class_summary}')
    lines.append('')
    lines.append(f'## Top {len(top)} poligonos pra revisar')
    lines.append('')
    lines.append('| # | geojson | poly_id | true | pred | prob_pred | prob_true | lon | lat |')
    lines.append('|---|---------|---------|------|------|-----------|-----------|-----|-----|')
    for i, s in enumerate(top, 1):
        row = (
            f"| {i} | `{s['geojson_source']}` | {s['polygon_id']} | "
            f"{s['true_class']} | **{s['pred_class']}** | "
            f"{s['prob_pred_class']:.3f} | {s['prob_true_class']:.3f} | "
            f"{s['centroid_lon']:.6f} | {s['centroid_lat']:.6f} |"
        )
        lines.append(row)
    lines.append('')
    lines.append('## Como revisar')
    lines.append('1. Abrir o geojson correspondente no QGIS.')
    lines.append('2. Localizar poligono pelo `polygon_id` (ou coords `lon/lat`).')
    lines.append('3. Decidir: **mover**, **redesenhar** ou **descartar**.')
    lines.append('4. Salvar geojson no Drive.')
    lines.append('5. Ao concluir o site, mover card pra `Aprovado`.')
    lines.append('')
    lines.append(f'CSV completo: `{SUSPECTS_CSV}` filtrado por `site={site}` e `kind={kind}`.')

    kind_lbl = ensure_label('misclass' if kind == 'misclass' else 'low-conf')
    description = '\n'.join(lines)
    issue = create_issue(title, description, [ciclo_lbl, run_lbl, susp_lbl, kind_lbl])
    print('criado:', issue['identifier'], issue['url'])
    created.append(issue)

print()
print('Total de cards criados:', len(created))

LINEAR_API_KEY: ··········
655 suspeitas carregadas do CSV
time= RIG state= Selecionadas
criado: RIG-397 https://linear.app/rigtech/issue/RIG-397/c1-doisriosflaviano-217-misclass-folha-estreita-folha-larga-116217
criado: RIG-398 https://linear.app/rigtech/issue/RIG-398/c1-giasa-203-misclass-folha-estreita-folha-larga-98203
criado: RIG-399 https://linear.app/rigtech/issue/RIG-399/c1-giasa-53-low-conf
criado: RIG-400 https://linear.app/rigtech/issue/RIG-400/c1-flaviano-47-misclass-folha-larga-folha-estreita-1747
criado: RIG-401 https://linear.app/rigtech/issue/RIG-401/c1-doisriosflaviano-47-low-conf
criado: RIG-402 https://linear.app/rigtech/issue/RIG-402/c1-celsoste2-44-misclass-folha-estreita-folha-larga-1644
criado: RIG-403 https://linear.app/rigtech/issue/RIG-403/c1-celso01-23-misclass-folha-larga-cultivo-1823
criado: RIG-404 https://linear.app/rigtech/issue/RIG-404/c1-celsoste2-14-low-conf
criado: RIG-405 https://linear.app/rigtech/issue/RIG-405/c1-flaviano-5-low-conf
criado: RIG-40

## 15. Próximo ciclo

1. Abrir `suspeitas_c1.csv` (ou QGIS pelas coords `centroid_lon/lat`) e revisar as manchas marcadas.
2. Corrigir no Drive: deletar polígono errado, ajustar geometria, mover pra outro geojson.
3. Bump `CICLO = 2` no topo, rodar tudo de novo.
4. **Cache é inteligente**: assinatura por site inclui `mtime` dos geojsons — só o site cujo geojson mudou recalcula features. Outros voltam do cache em segundos.
5. Comparar `relatorio_c2.txt` vs `_c1`: F1 daninha subiu? Recall subiu sem perder precision? Número de suspeitas caiu? → dataset melhorou.

**Regra**: se o modelo é bem calibrado (mesmo instrumento, mesmos params), qualquer melhora vem de rótulos melhores — não de tuning.

In [4]:
import os

print(os.listdir('/content/drive/MyDrive'))

FileNotFoundError: [Errno 2] No such file or directory: '/content/drive/MyDrive'